In [3]:
import json
from pathlib import Path
from collections import Counter
import bisect


def load_merged_track(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def exact_or_nearest_index(times, query_time_sec, allow_nearest=True):
    """
    If exact sampled time exists, use it.
    Otherwise use nearest sampled time if allow_nearest=True.
    """
    times = [float(t) for t in times]
    idx = bisect.bisect_left(times, float(query_time_sec))

    if idx < len(times) and abs(times[idx] - float(query_time_sec)) < 1e-6:
        return idx

    if not allow_nearest:
        return None

    if idx == 0:
        return 0
    if idx >= len(times):
        return len(times) - 1

    before = idx - 1
    after = idx
    if abs(times[before] - float(query_time_sec)) <= abs(times[after] - float(query_time_sec)):
        return before
    return after


def get_states_at_time(merged_data, query_time_sec, allow_nearest=True):
    """
    Returns one row per object:
      assoc_id, name, sampled_time, status, projected_pixel, world_coordinates
    """
    object_tracks = merged_data.get("object_tracks", {})
    if not object_tracks:
        raise ValueError("No 'object_tracks' found in merged JSON")

    results = []

    for assoc_id, tr in object_tracks.items():
        times = tr.get("sampled_times_sec", [])
        if not times:
            continue

        idx = exact_or_nearest_index(times, query_time_sec, allow_nearest=allow_nearest)
        if idx is None:
            continue

        results.append({
            "assoc_id": assoc_id,
            "name": tr.get("name", assoc_id),
            "sampled_time": float(times[idx]),
            "status": tr.get("status_samples", [None] * len(times))[idx],
            "projected_pixel": tr.get("projected_pixel_samples", [None] * len(times))[idx],
            "world_coordinates": tr.get("world_coordinate_samples", [None] * len(times))[idx],
        })

    return results


def print_states(states, sort_by="status"):
    if sort_by == "status":
        states = sorted(states, key=lambda x: (str(x["status"]), str(x["name"])))
    elif sort_by == "name":
        states = sorted(states, key=lambda x: str(x["name"]))

    print("\n=== Object states at query time ===")
    for s in states:
        print(f"{s['assoc_id']} | {s['name']} | {s['status']}")
        # print(
        #     f"{s['assoc_id']} | {s['name']} | "
        #     f"t={s['sampled_time']:.1f} | status={s['status']} | "
        #     f"pixel={s['projected_pixel']} | world={s['world_coordinates']}"
        # )


def print_summary(states):
    counter = Counter(s["status"] for s in states)
    print("\n=== Status summary ===")
    for status, count in sorted(counter.items(), key=lambda x: (-x[1], str(x[0]))):
        print(f"{status}: {count}")


# ---- usage ----
merged_path = "scripts/staged_oos_vqa_generation/object_object_relation/outputs/merged_tracks/P04/P04-20240413-142619_merged_visibility_track.json"
query_time = 61.0

data = load_merged_track(merged_path)
states = get_states_at_time(data, query_time, allow_nearest=True)
print_summary(states)
print_states(states, sort_by="status")

FileNotFoundError: [Errno 2] No such file or directory: 'scripts/staged_oos_vqa_generation/object_object_relation/outputs/merged_tracks/P04/P04-20240413-142619_merged_visibility_track.json'

In [4]:
# ---- usage ----
merged_path = "/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/scripts/staged_oos_vqa_generation/object_object_relation/outputs/merged_tracks/P04/P04-20240413-142619_merged_visibility_track.json"
query_time = 61.0

data = load_merged_track(merged_path)
states = get_states_at_time(data, query_time)
print_states(states)


=== Object states at query time ===
24a180dd40d2d5f1 | measuring cup | geometrically_occluded
ed756fd25589eb34 | box of chicken | in_motion
a956ffc59c8a15da | black jar1 | observed_not_visible_in_open_fixture
7457a62c58c523e8 | black jar2 | observed_not_visible_in_open_fixture
a6d945305f4c8100 | bottle of oil | observed_not_visible_in_open_fixture
3ffec4b711151386 | condiment jar | observed_not_visible_in_open_fixture
e07e73b7e5188d9c | herb jar1 | observed_not_visible_in_open_fixture
a192a0897d1b82b4 | herb jar2 | observed_not_visible_in_open_fixture
c91abd3e2d1f2844 | herb jar3 | observed_not_visible_in_open_fixture
95dc26bbfd03ff97 | spice jar1 | observed_not_visible_in_open_fixture
3f070f24df5ec151 | spice jar2 | observed_not_visible_in_open_fixture
b6a5cedf2cf5ebcd | spice jar3 | observed_not_visible_in_open_fixture
8a765b15984b0e82 | spoon2 | observed_not_visible_in_open_fixture
98ffa9ff7f44fe0d | spoon3 | observed_not_visible_in_open_fixture
5d8e61b3f5c17472 | weighing scales |